# H7 Content use (select-one and the notes audit, for now)

Frozen before the run.

**Claim.** Answers depend on what the workers sent. When the orchestrator has both text and cache, it follows the text.

**Arms** (thinking off; AB and BA; seed 0):
- K0 with both workers.
- K0 with only worker A's block.
- K0 with only worker B's block.

**Notes audit:** read from the H5 scan rows. Count how often the note written after the second worker's block repeats the first note: the orchestrator is ignoring the new cache and copying its own text.

**Prediction.**
- On 2-hop questions, both-workers > each single worker, and the hop-2 worker alone > the hop-1 worker alone (the hop-2 paragraph often names the answer).
- On toy hop1 and hop2, each single worker is near 0.
- In the notes audit, note 2 repeats note 1 on a large share of questions.

**Pilot (Qwen2.5-7B, MuSiQue 2-step, n = 200, EM):**
- Step-1 worker alone 14, step-2 worker alone 30, both 51.
- Notes audit: note 2 identical to note 1 on 45/200, and >80% the same on another 92/200.

**Not built yet:**
- Evidence edit, split by closed-book correctness.
- Payload swap across questions.
- Thought swap (K1).
- The text-vs-cache conflict test (v7).

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

In [ ]:
qs = [q for name in ("toy", "toy_d0", "musique") for q in data.load(name, seed=0, placement=["AB", "BA"])]
if SMOKE:
    qs = data.per_cell(qs, 2)
print(len(qs), "questions")

ARMS = {
    "k0": (arms.k0, {}),
    "k0_only_a": (arms.k0, {"visible": [0]}),
    "k0_only_b": (arms.k0, {"visible": [1]}),
}

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df, by=("dataset", "qtype", "placement"))

## Both workers vs one

In [ ]:
for one in ("k0_only_a", "k0_only_b"):
    print(f"k0 − {one}")
    display(analysis.paired(df, "k0", one, by=("dataset", "qtype", "placement")))

## Notes audit (H5's scan rows)

Same questions as `h5_rounds.ipynb`; run that notebook first.

In [ ]:
from difflib import SequenceMatcher

SPLIT = ["AB", "BA", "AAB", "ABA", "AABB", "ABBA", "ABAB"]     # rounds have nothing to repair at 0 crossings
QTYPES = ["hop1", "hop2", "2hop", "3hop1", "4hop1"]           # toy hop0 is 0 crossings at every placement
h5_qs = [q for name in ("toy", "toy_d0", "musique") for q in data.load(name, seed=0, placement=SPLIT, qtype=QTYPES)]
h5_qs = data.per_cell(h5_qs, 2 if SMOKE else 300)
H5_SCAN = {"scan_1": (arms.scan, {"rounds": 1}), "scan_2": (arms.scan, {"rounds": 2})}   # as in h5_rounds
scan = load_results(h5_qs, H5_SCAN, MODEL, DTYPE)
rows = []
for arm, g in scan.groupby("arm"):
    pairs = [n[:2] for n in g.notes if len(n) >= 2]
    if pairs:
        same = sum(a.strip() == b.strip() for a, b in pairs)
        near = sum(a.strip() != b.strip() and SequenceMatcher(None, a, b).ratio() > 0.8 for a, b in pairs)
        rows.append({"arm": arm, "n": len(pairs), "note 2 == note 1": same, "note 2 >80% similar": near})
pd.DataFrame(rows)

## Findings

_(fill in after the full run)_